# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.36702465 -0.84918891 -0.13612599 -0.57146028  0.58567354]
 [-0.44483146  0.05183434 -0.48589003 -0.75788452 -0.21380554]
 [ 0.66064628 -0.91794058 -0.6573124  -0.95100795  0.5011209 ]
 [ 0.07706762 -0.20392151  0.39099552  0.71061918  0.62945817]
 [ 0.20215294 -0.00224427  0.76079564  0.83099187  0.80722402]
 [ 0.28666522 -0.81237201  0.82866257 -0.91628203 -0.07988366]
 [-0.20035062 -0.35099085 -0.10003091  0.35504183  0.47720833]
 [-0.89492622 -0.8547006   0.80475207 -0.65341102  0.99016116]
 [-0.6996793   0.93844659 -0.30279567 -0.68352908 -0.65784717]
 [-0.26939649  0.99824961 -0.87126126  0.86352375  0.04643973]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a2', 'a1', 'a2', 'a1', 'a1', 'a2', 'a2', 'a1', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 1, 1, 1, 0, 1, 1, 0, 1, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:33,  1.03s/it]

SVI:   3%|▎         | 1/34 [00:01<00:33,  1.03s/it, loss=2754.8665]

SVI:   6%|▌         | 2/34 [00:01<00:32,  1.03s/it, loss=3647.8027]

SVI:   9%|▉         | 3/34 [00:01<00:31,  1.03s/it, loss=3020.5388]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.16it/s, loss=2100.3125]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.16it/s, loss=2595.5723]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.16it/s, loss=2679.7854]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.96it/s, loss=2679.7854]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.96it/s, loss=2009.9371]